# Audit: ProductCategory as evidence and a Segment baseline

Test the quoted claim against the supplied CSVs: completeness, retailer-specific paths, variable depth, malformed/duplicated paths, ambiguous Segment mappings, and the usefulness of 90%+ prefix rules. The target has no Segment ground truth, so accuracy is measured on held-out training products only. Source files are never modified.

Repository context: README.md requests supporting product evidence; docs/findings/00_data_profile/02_cleaning_direction.md section 12 documents ProductCategory quality concerns. Whether this is the *most natural* evidence field is a judgment, not a measurable completeness claim. Category is retailer-supplied context, not independent proof of a correct Segment.


## Results from this run

Observed completeness: training **99.9657%**, target **99.9751%**.

The fixed 90%-purity, minimum-20-row rules cover **84.16%** of held-out labeled rows, with **95.74%** accuracy among covered rows. The majority baseline on the same covered rows scores **67.65%**. Target coverage is **82.70%**, with no target accuracy claim.

Use the quality, ambiguity, and per-class tables below to assess the qualifications in the quoted claim. These results support a measured decision about breadcrumb rules rather than assuming training purity guarantees correctness.


## Verdict on the quoted claim

- **Completeness confirmed:** 99.9657% and 99.9751% round to the claimed 99.97% training and 99.98% target.
- **?One of the few? is overstated without a narrower definition of useful fields:** 16 of 32 training fields and 14 of 32 target fields are at least 99% populated. Completeness alone does not establish predictive usefulness.
- **Retailer variation supported:** all 518 observed UPC groups spanning multiple retailers have different normalized paths. These are candidate identity matches, not independently verified products.
- **Quality concerns supported:** each dataset has one exact repeated breadcrumb sequence and one single-node value requiring inspection. The notebook also reports depth variation. These simple checks are not an exhaustive malformed-record detector.
- **No universal one-to-one mapping:** 447 retailer/full-path groups have multiple supplied Segment labels, covering 67,113 of 79,750 eligible labeled rows. Annotation noise could explain some conflicts.
- **Useful baseline supported in this split:** 95.74% selective validation accuracy at 84.16% coverage, versus 67.65% majority accuracy on the same covered rows. This is an empirical result, not a guarantee that every 90%-pure prefix generalizes.
- **Supporting evidence is reasonable; ?most natural? remains subjective:** a breadcrumb can explain retailer context. Broad Pain Relievers categories require inspecting the full path and the observed Segment distribution; they do not independently prove Internal Analgesics.


In [1]:
from pathlib import Path
import pandas as pd
import hashlib
import re
ROOT = Path.cwd()
if not (ROOT / 'data' / 'provided').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'provided'
train = pd.read_csv(DATA / 'Selfcare_Training_data (1).csv', low_memory=False)
target = pd.read_csv(DATA / 'Selfcare_Target_data (1).csv', low_memory=False)
def present(s): return s.notna() & s.astype('string').str.strip().ne('').fillna(False)
def tokens(value):
    if pd.isna(value): return ()
    return tuple(re.sub(r'\s+', ' ', x).strip().casefold() for x in str(value).split('>') if x.strip())
def repeated(t):
    return bool(t) and any(len(t) % k == 0 and t == t[:k] * (len(t)//k) for k in range(1, len(t)))
def collapse(t):
    for k in range(1, len(t)):
        if len(t) % k == 0 and t == t[:k] * (len(t)//k): return t[:k]
    return t


## 1. Verify the exact completeness percentages and compare other fields

In [2]:
rows=[]
for name, df in [('training', train), ('target', target)]:
    mask=present(df['ProductCategory'])
    rows.append({'dataset':name, 'rows':len(df), 'nonnull':df['ProductCategory'].notna().sum(),
                 'nonblank':mask.sum(), 'populated_pct':100*mask.mean(), 'missing_or_blank':int((~mask).sum())})
    coverage=df.apply(lambda s: 100*present(s).mean()).sort_values(ascending=False)
    print('\n'+name+' field completeness (%)')
    print(coverage.round(4).to_string())
    print('Fields >=99% populated:', int(coverage.ge(99).sum()), 'of', len(coverage))
completeness=pd.DataFrame(rows)
print('\nClaim check (round to two decimals):')
print(completeness.to_string(index=False))
print('Claimed training 99.97%, observed:', round(completeness.iloc[0]['populated_pct'],2))
print('Claimed target 99.98%, observed:', round(completeness.iloc[1]['populated_pct'],2))



training field completeness (%)
JoiningKey             100.0000
Retailer               100.0000
Category                99.9988
ProductRating           99.9953
ReviewsCount            99.9953
XRatXRev                99.9941
Sku                     99.9929
ProductCategory         99.9657
MDM_Id                  99.9219
ProductBrand            99.8817
MDM_InsertDateTime      99.8604
ProductReviewsCount     99.8356
ProductName             99.8214
ProductUrl              99.7303
Brand                   99.6192
ProductImageUrl         99.1082
Segment                 94.3360
Mnfr                    93.2610
TargetAgeGroup          91.6300
Sub-Segment             91.1120
ProductContents         39.8666
ProductDescription      32.2488
Upc                     20.1947
ProductModelNumber      14.1771
Exclude                  6.2648
Platform                 1.3069
Sun1                     0.0035
Sun2                     0.0024
Sun4                     0.0024
Sun5                     0.0024
Sun3   

## 2. Breadcrumb depth and quality

Whitespace and case are normalized for comparison. Exact whole-path repetitions are collapsed for modeling. Empty components and single-node values are review flags, not automatic errors; legitimate flat categories exist. These checks cannot detect every malformed path.


In [3]:
for name, df in [('training', train), ('target', target)]:
    raw=df['ProductCategory'].map(tokens)
    df['_path']=raw.map(collapse)
    df['_retailer']=df['Retailer'].astype('string').fillna('<missing>').str.strip().str.casefold()
    flags=pd.DataFrame({'missing':~present(df['ProductCategory']), 'single_node':raw.map(len).eq(1),
                        'exact_repeated_path':raw.map(repeated),
                        'empty_component':df['ProductCategory'].map(lambda v: False if pd.isna(v) else any(not x.strip() for x in str(v).split('>')))})
    print('\n'+name+' quality flags (may overlap):'); print(flags.sum().to_string())
    depth=raw.map(len)
    print('Depth distribution:'); print(depth.value_counts().sort_index().to_string())
    print('Depth by retailer:'); print(df.assign(depth=depth).groupby('Retailer')['depth'].agg(['count','min','median','max']).to_string())
    for flag in ['exact_repeated_path','empty_component','single_node']:
        print('Examples:',flag)
        print(df.loc[flags[flag],['Retailer','ProductCategory']].head(5).to_string(index=False,max_colwidth=170))



training quality flags (may overlap):
missing                29
single_node             1
exact_repeated_path     1
empty_component         0
Depth distribution:
ProductCategory
0       29
1        1
2     1090
3    23453
4    38788
5    14012
6     6533
7      644
8        1
9        1
Depth by retailer:
                                                                        count  min  median  max
Retailer                                                                                       
 19 amino acids nourish the skins dermal layers; and Vitamin C              2    0     0.5    1
 Magnesium stearate."                                                       1    0     0.0    0
 and supports healthy joints* SUGGESTED USE FOR MONTHLY SUPPLY: 30-day      1    0     0.0    0
Amazon                                                                  66001    0     4.0    8
CVS                                                                      2329    2     3.0    4
Kroger              

## 3. Same product across retailers

A shared nonblank UPC is a candidate product match, not verified identity. Count UPCs appearing at multiple retailers and compare normalized paths. Bad UPCs, variants, and annotation mistakes can affect these results.


In [4]:
upc=train.loc[present(train['Upc']) & train['_path'].map(bool)].copy()
upc['_upc']=upc['Upc'].astype('string').str.strip()
by_upc=upc.groupby('_upc').agg(retailers=('_retailer','nunique'), paths=('_path','nunique'))
cross=by_upc[by_upc['retailers'].gt(1)]
print('UPCs across multiple retailers:',len(cross))
print('Those with different normalized paths:',int(cross['paths'].gt(1).sum()))
example_keys=cross.loc[cross['paths'].gt(1)].head(5).index
print(upc.loc[upc['_upc'].isin(example_keys),['Upc','Retailer','ProductName','ProductCategory','Segment']].drop_duplicates().head(30).to_string(index=False,max_colwidth=110))


UPCs across multiple retailers: 518
Those with different normalized paths: 518
         Upc Retailer                                                                                                    ProductName                                                                          ProductCategory                          Segment
025715690186  Walmart                                                                Flents Quiet Please Earplugs, 100 Pair, NRR 29,                                                  Health and Medicine > Ear Care > Flents                    Lifestyle CHC
023249012726  Walmart                                                            Sports Research Multi Collagen Complex, 90 Capsules                          Health and Medicine>Vitamins and Supplements>Beauty Supplements Vitamins, Minerals & Supplements
027917005300  Walmart                                                                                         Melatonin Free Bedtime                        

## 4. Test whether a complete breadcrumb determines Segment

Use available nonblank Segment labels as supplied, without claiming they are verified ground truth. Report full-path ambiguity within retailer; this avoids treating unrelated retailer vocabularies as one taxonomy. Inspect Pain Relievers explicitly: a broad pain category can include external and internal products.


In [5]:
labeled=train.loc[present(train['Segment']) & train['_path'].map(bool)].copy()
labeled['_segment']=labeled['Segment'].astype('string').str.strip()
counts=labeled.groupby(['_retailer','_path','_segment']).size().rename('n').reset_index()
path_stats=counts.groupby(['_retailer','_path']).agg(rows=('n','sum'),segments=('_segment','nunique'),majority=('n','max'))
path_stats['purity']=path_stats['majority']/path_stats['rows']
print('Retailer/path groups:',len(path_stats))
print('Groups with multiple Segments:',int(path_stats['segments'].gt(1).sum()))
print('Rows in ambiguous groups:',int(path_stats.loc[path_stats['segments'].gt(1),'rows'].sum()),'of',len(labeled))
print('Largest ambiguous groups:')
print(path_stats.loc[path_stats['segments'].gt(1)].sort_values('rows',ascending=False).head(15).to_string())
pain=labeled['_path'].map(lambda t: any('pain reliever' in x for x in t))
print('\nSegments for paths containing Pain Relievers:')
print(labeled.loc[pain,'_segment'].value_counts().to_string())
print('Segments for paths ENDING in Pain Relievers:')
print(labeled.loc[labeled['_path'].map(lambda t: bool(t) and 'pain reliever' in t[-1]),'_segment'].value_counts().to_string())


Retailer/path groups: 1344
Groups with multiple Segments: 447
Rows in ambiguous groups: 67113 of 79750
Largest ambiguous groups:
                                                                                                                               rows  segments  majority    purity
_retailer _path                                                                                                                                                  
amazon    (health & household, vitamins, minerals & supplements, blended vitamin & mineral supplements)                        8311         7      7991  0.961497
          (health & household, vitamins, minerals & supplements, vitamins, multivitamins)                                      2064         4      2015  0.976260
          (health & household, vitamins, minerals & supplements, collagen)                                                     1779         3      1764  0.991568
          (health & household, vitamins, minerals & supplemen

## 5. Hold out product groups before learning rules

Connect rows sharing a nonblank UPC, product URL, or normalized exact product name, including transitive matches. Hash each resulting group deterministically into an approximately 80/20 fit/validation split. This reduces duplicate leakage across retailers; different names/identifiers may still describe the same product. Bad shared identifiers may over-group records. No target labels are used.

Rules are retailer-specific prefixes, with at least 20 fit rows and at least 90% majority agreement. These thresholds are fixed before evaluating. Select the deepest qualifying prefix; ties in class counts use lexical order. Abstain if no qualifying prefix exists. Compare against a fit-set majority baseline and report both selective accuracy and coverage; training purity is not validation accuracy.


In [6]:
parent=list(range(len(train)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[max(a,b)]=min(a,b)
for col in ['Upc','ProductUrl','ProductName']:
    seen={}
    for i,value in train[col].items():
        if pd.isna(value): continue
        key=re.sub(r'\s+',' ',str(value)).strip().casefold()
        if not key: continue
        if key in seen: union(i,seen[key])
        else: seen[key]=i
labeled['_group']=[find(i) for i in labeled.index]
is_validation=labeled['_group'].map(lambda x: int(hashlib.sha256(str(x).encode()).hexdigest()[:8],16)%5==0)
fit=labeled.loc[~is_validation].copy(); validation=labeled.loc[is_validation].copy()
assert set(fit['_group']).isdisjoint(validation['_group'])
print('Fit / validation rows:',len(fit),len(validation))
print('Fit / validation product groups:',fit['_group'].nunique(),validation['_group'].nunique())
from collections import Counter, defaultdict
prefix_counts=defaultdict(Counter)
for retailer,path,segment in fit[['_retailer','_path','_segment']].itertuples(index=False,name=None):
    for depth in range(1,len(path)+1): prefix_counts[(retailer,path[:depth])][segment]+=1
rules={}
for key,counter in prefix_counts.items():
    support=sum(counter.values()); winner=sorted(counter,key=lambda s:(-counter[s],s))[0]
    purity=counter[winner]/support
    if support>=20 and purity>=0.9: rules[key]=(winner,support,purity)
def predict(retailer,path):
    for depth in range(len(path),0,-1):
        key=(retailer,path[:depth])
        if key in rules: return (*rules[key],depth,' > '.join(path[:depth]))
    return (None,0,float('nan'),0,None)
def apply_rules(df):
    return pd.DataFrame([predict(r,p) for r,p in df[['_retailer','_path']].itertuples(index=False,name=None)],
                        index=df.index,columns=['prediction','fit_support','fit_purity','depth','evidence_prefix'])
vp=apply_rules(validation)
covered=vp['prediction'].notna(); correct=vp['prediction'].eq(validation['_segment'])
majority=fit['_segment'].value_counts().index[0]
metrics={'qualifying_prefixes':len(rules),'validation_rows':len(validation),
         'coverage_pct':100*covered.mean(),'accuracy_on_covered_pct':100*correct[covered].mean(),
         'correct_and_covered_pct':100*correct.mean(),
         'majority_baseline_all_pct':100*validation['_segment'].eq(majority).mean(),
         'majority_baseline_covered_pct':100*validation.loc[covered,'_segment'].eq(majority).mean(),
         'covered_disagreements':int((covered & ~correct).sum())}
print(pd.Series(metrics).to_string())
report=validation[['_retailer','_segment','_group']].join(vp)
report['covered']=covered; report['correct']=correct
print('\nPer-retailer performance:')
print(report.groupby('_retailer').apply(lambda g: pd.Series({'rows':len(g),'coverage_pct':100*g.covered.mean(),
 'covered_accuracy_pct':100*g.loc[g.covered,'correct'].mean()})).to_string())
print('\nPer-Segment coverage and accuracy:')
print(report.groupby('_segment').apply(lambda g: pd.Series({'rows':len(g),'coverage_pct':100*g.covered.mean(),
 'covered_accuracy_pct':100*g.loc[g.covered,'correct'].mean()})).to_string())
print('\nHeld-out disagreements: review candidates, not confirmed errors')
print(validation.loc[covered & ~correct,['Retailer','ProductName','Segment','ProductCategory']].join(vp).head(15).to_string(index=False,max_colwidth=100))


Fit / validation rows: 63837 15913
Fit / validation product groups: 59475 14602
qualifying_prefixes                356.000000
validation_rows                  15913.000000
coverage_pct                        84.157607
accuracy_on_covered_pct             95.736260
correct_and_covered_pct             95.736260
majority_baseline_all_pct           61.710551
majority_baseline_covered_pct       67.652330
covered_disagreements              571.000000

Per-retailer performance:
                                                                  rows  coverage_pct covered_accuracy_pct
_retailer                                                                                                
19 amino acids nourish the skins dermal layers; and vitamin c      1.0      0.000000                 <NA>
amazon                                                         12545.0     87.046632            95.879121
cvs                                                              418.0     74.401914            96.78

## 6. Target coverage and evidence examples

Apply the same fit-only rules to the target. Target coverage measures applicability of those rules, not accuracy or generalization to entirely new products. Prefix purity is an empirical fit proportion, not calibrated model confidence. A contradiction with a future model prediction would be a review signal; the examples above compare rules with supplied labels instead.


In [7]:
tp=apply_rules(target)
print('Target rows:',len(target))
print('Target rule coverage (%):',100*tp['prediction'].notna().mean())
print(target[['Retailer','ProductName','ProductCategory']].join(tp).loc[tp['prediction'].notna()].head(12).to_string(index=False,max_colwidth=110))
print('\nEvidence templates (retailer context, not a causal guarantee):')
for idx in tp.index[tp['prediction'].notna()][:5]:
    row=tp.loc[idx]
    print(f"{target.loc[idx,'Retailer']} categorizes this product under {row.evidence_prefix}. "
          f"In fitting data, {row.fit_purity:.1%} of {row.fit_support} rows under this prefix have Segment {row.prediction}.")


Target rows: 28082
Target rule coverage (%): 82.69710134605798
Retailer                                                                                                    ProductName                                                                                                ProductCategory                       prediction  fit_support  fit_purity  depth                                                                               evidence_prefix
  Amazon Michael's Naturopathic Programs Kidney Factors - 120 Vegetarian Tablets - Nutrients for Kidney Function & C...                  Health & Household > Vitamins, Minerals & Supplements > Blended Vitamin & Mineral Supplements Vitamins, Minerals & Supplements         6682    0.960192      3 health & household > vitamins, minerals & supplements > blended vitamin & mineral supplements
  Amazon Sustainably US Grown, Organic Barley Grass Powder, 20 Ounce (1.25 Pounds), Rich Fibers, Immune Vitamins, Mi...                       Health & Househ

## Interpretation limits

- Presence is not validity: near-complete breadcrumbs may still be malformed, broad, promotional, or inconsistent.
- Ambiguous paths disprove a universal one-to-one mapping only relative to supplied labels; some ambiguity may reflect label errors.
- Evaluate the 90% rule using held-out accuracy and coverage together. The baseline may work well on dominant classes but fail or abstain on rare ones.
- ProductCategory is useful human-readable supporting context. The claim that it is the best evidence field requires comparison with product names/descriptions and human review.
- The experiment is a single deterministic group split, not a production benchmark. Threshold tuning requires a separate validation strategy; do not repeatedly optimize against this holdout.
